### imports

In [1]:
import os
import subprocess
import re
import json
import warnings
from typing import TypedDict, List, Dict, Any, Optional, Literal


import speech_recognition as sr
import pyttsx3

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_tavily import TavilySearch, TavilyExtract

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

warnings.filterwarnings("ignore")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_996\1404238845.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

### installation

### EXpermentation

#### search engine informal RAG

In [3]:
tavily_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)


In [4]:
x="give command to open cammera"

In [5]:
result = tavily_search.invoke({
    "query": x,
    "include_domains": [
        "learn.microsoft.com"
    ]
})

In [6]:
print(result)

{'query': 'give command to open cammera', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://learn.microsoft.com/en-us/answers/questions/4280402/how-do-i-make-camera-exe-start-with-a-command', 'title': 'How do i make camera.exe start with a command - Microsoft Learn', 'content': '0 comments No comments\n\nAnswer accepted by question author\n\nAnonymous\n\n2022-11-20T02:49:02+00:00\n\nHello Aiva Buldere\n\nWelcome to Microsoft Community.\n\nI am aware of the present situation is that you need to launch your camera application in the form of command. Generally speaking, Camera is a built-in UWP application of Windows, so in essence, it cannot be started through simple camera.exe\n\nThe following commands are used to start the built-in camera application in two ways. I hope they can be of reference to you:\n\n|  Press the Windows key+R at the same time to display the Run command. typing  microsoft.windows.camera：  And press OK. This will immediately op

In [7]:
urls = [
    item["url"]
    for item in result["results"]
]

In [8]:
from langchain_tavily import TavilyExtract

tavily_extract = TavilyExtract()

extracted = tavily_extract.invoke({
    "urls": urls[:3]
})

print(extracted)

{'results': [{'url': 'https://learn.microsoft.com/en-us/answers/questions/4280402/how-do-i-make-camera-exe-start-with-a-command', 'title': 'How do i make camera.exe start with a command - Microsoft Learn', 'raw_content': "This browser is no longer supported.\n\nUpgrade to Microsoft Edge to take advantage of the latest features, security updates, and technical support.\n\n[Download Microsoft Edge](https://go.microsoft.com/fwlink/p/?LinkID=2092881 )   [More info about Internet Explorer and Microsoft Edge](https://learn.microsoft.com/en-us/lifecycle/faq/internet-explorer-microsoft-edge)\n\n\n\n# How do i make camera.exe start with a command\n\nI'm making a couple hotkeys for my computer that launch apps, but i cannot figure out how to make the camera app start with a command. I'm using HIDmacros for the hotkeys\n\n[![](https://learn-attachment.microsoft.com/api/attachments/033801dd-f53b-4ae6-a8ee-f65dcaa99ee2?platform=QnA)](https://learn-attachment.microsoft.com/api/attachments/033801dd-f

In [9]:
def find_command_results(result):
    command_results = []

    for item in result["results"]:
        text = item.get("raw_content", "")

        if any(keyword in text.lower() for keyword in [
            "command",
            "cmd",
            "command prompt",
            "powershell",
            "terminal",
            "run this",
            "type the following"
        ]):
            command_results.append({
                "title": item["title"],
                "url": item["url"],
                "content": text
            })

    return command_results

In [10]:
find_command_results(extracted)

[{'title': 'How do i make camera.exe start with a command - Microsoft Learn',
  'url': 'https://learn.microsoft.com/en-us/answers/questions/4280402/how-do-i-make-camera-exe-start-with-a-command',
  'content': "This browser is no longer supported.\n\nUpgrade to Microsoft Edge to take advantage of the latest features, security updates, and technical support.\n\n[Download Microsoft Edge](https://go.microsoft.com/fwlink/p/?LinkID=2092881 )   [More info about Internet Explorer and Microsoft Edge](https://learn.microsoft.com/en-us/lifecycle/faq/internet-explorer-microsoft-edge)\n\n\n\n# How do i make camera.exe start with a command\n\nI'm making a couple hotkeys for my computer that launch apps, but i cannot figure out how to make the camera app start with a command. I'm using HIDmacros for the hotkeys\n\n[![](https://learn-attachment.microsoft.com/api/attachments/033801dd-f53b-4ae6-a8ee-f65dcaa99ee2?platform=QnA)](https://learn-attachment.microsoft.com/api/attachments/033801dd-f53b-4ae6-a8e

In [11]:
## combined

In [12]:
user_input = input("enter: ")

tavily_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced")

result = tavily_search.invoke({
    "query": f"Windows cmd command for: {user_input}",
    "include_domains": [
        "learn.microsoft.com"
    ]
})

urls = [
    item["url"]
    for item in result["results"]
]


tavily_extract = TavilyExtract()

extracted = tavily_extract.invoke({
    "urls": urls[:3]
})


extracted=find_command_results(extracted)
joined=chr(10).join(str(result) for result in extracted)
print(joined)

enter:  open paint


{'title': 'Microsoft Paint - Microsoft Q&A', 'url': 'https://learn.microsoft.com/en-us/answers/questions/3226768/microsoft-paint', 'content': 'This browser is no longer supported.\n\nUpgrade to Microsoft Edge to take advantage of the latest features, security updates, and technical support.\n\n# Microsoft Paint\n\n![](data:image/svg+xml, %3Csvg xmlns=\'http://www.w3.org/2000/svg\' height=\'64\' class=\'font-weight-bold\' style=\'font: 600 30.11764705882353px "SegoeUI", Arial\' width=\'64\'%3E%3Ccircle fill=\'hsl(128, 8%, 22%)\' cx=\'32\' cy=\'32\' r=\'32\' /%3E%3Ctext x=\'50%25\' y=\'55%25\' dominant-baseline=\'middle\' text-anchor=\'middle\' fill=\'%23FFF\' %3EA%3C/text%3E%3C/svg%3E)\n\nI see some people are having problems with Microsoft Paint.\xa0 I have a BIG problem!!!\n\nI am not computer savvy.\xa0 I know just enough to be dangerous.\xa0 Sometimes if I poke around I can find a solution to a problem.\n\nFirst, I didn\'t know I had Paint.\xa0 Supposedly it comes with Windows 10.\n

#### llms

In [13]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-235B-A22B-Instruct-2507",
    temperature=0.1,
    max_new_tokens=512,
    do_sample=False,
)

llm = ChatHuggingFace(
    llm=llm_endpoint
)

In [14]:
type(extracted)

list

In [15]:
joined=chr(10).join(str(result) for result in extracted)

In [16]:
prompt = f"""
You are Agent, an autonomous Windows operating-system command generator.

Your job is to convert the user's natural-language request into the
single Windows command that should be executed.

USER REQUEST:
{x}

RETRIEVED WINDOWS DOCUMENTATION:
{joined}

IMPORTANT:
The retrieved documentation is supplementary context, not the only source
of knowledge. Use your own pretrained knowledge of Windows, CMD,
PowerShell, Windows applications, URI schemes, executables, and
operating-system behavior together with the retrieved documentation.

Determine the correct command for the user's request.

OUTPUT REQUIREMENTS:
- Output ONLY the final executable command.
- Do NOT explain your reasoning.
- Do NOT output <think> tags.
- Do NOT describe the command.
- Do NOT output "Command:".
- Do NOT output Markdown.
- Do NOT output code fences.
- Do NOT output multiple alternatives.
- Do NOT repeat the user's request.
- Return exactly ONE command that can be passed to the Windows operating system.

If the retrieved documentation does not contain the answer, use your
own knowledge to determine the correct Windows command.

USER REQUEST:
{x}
"""

response = llm.invoke(prompt)

command = response.content.strip()

print(command)

start microsoft.windows.camera:


### workflow

In [17]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-235B-A22B-Instruct-2507",
    temperature=0.1,
    max_new_tokens=512,
    do_sample=False,
)

llm = ChatHuggingFace(
    llm=llm_endpoint
)

In [18]:
# STATE
class AgentState(TypedDict, total=False):

    user_input: str

    intent: Literal[
        "END",
        "COMMAND",
        "CONVERSATION"
    ]

    rag_results: str

    command: str

    verification_required: bool

    verification_response: str

    verification_decision: Literal[
        "execution",
        "cancel"
    ]

    execution_result: str

    response: str

    history: List[Any]

In [38]:
def speak(text: str):
    recognizer = sr.Recognizer()
    engine = pyttsx3.init()
    
    engine.setProperty("rate", 145)
    engine.setProperty("volume", 1)

    print("Agent TK:", text)

    engine.say(text)
    engine.runAndWait()


In [38]:
engine = pyttsx3.init()

engine.setProperty("rate", 145)
engine.setProperty("volume", 1)


def speak(text: str):

    print("Agent:", text)

    engine.say(text)
    engine.runAndWait()


In [39]:
def listen() -> str:
    
    recognizer = sr.Recognizer()

    with sr.Microphone() as source:

        print("\nListening...")

        recognizer.adjust_for_ambient_noise(
            source,
            duration=0.5
        )

        audio = recognizer.listen(source)

    try:

        query = recognizer.recognize_google(audio)

        print("You:", query)

        return query.strip()

    except sr.UnknownValueError:

        speak("I could not understand that.")

        return ""

    except sr.RequestError:

        speak("Speech recognition service is unavailable.")

        return ""


In [40]:
def User_node(state: AgentState):
    # tkae userinput  
    # desides wheather it is COMMAND, CONVERSATION or END

    user_input = listen()

    history = state.get("history", [])

    if not user_input:
        return {
            "user_input": "",
            "history": history,
            "intent": "CONVERSATION"
        }

    history.append(
        HumanMessage(content=user_input)
    )

    prompt = f"""
You are the routing controller for a Windows voice automation agent.

Classify the user's latest request into exactly one category.

END:
The user wants to stop, exit, quit, or terminate the agent.

COMMAND:
The user wants the agent to perform an operation on Windows,
applications, files, folders, settings, processes, browser,
terminal, or another system resource.

CONVERSATION:
The user is having normal conversation, asking a question,
greeting, or saying something that does not require a
Windows operating-system action.

USER REQUEST:
{user_input}

Return ONLY one word:

END
COMMAND
CONVERSATION
"""

    response = llm.invoke(prompt)

    result = response.content.strip().upper()

    if result == "END":
        intent = "END"

    elif result == "COMMAND":
        intent = "COMMAND"

    else:
        intent = "CONVERSATION"

    return {
        "user_input": user_input,
        "history": history,
        "intent": intent
    }

In [41]:
def route_request(
    state: AgentState
) -> Literal[
    "end",
    "command",
    "conversation"
]:

    intent = state.get("intent")

    if str.lower(intent) == str.lower("END"):
        return "end"

    if str.lower(intent) == str.lower("COMMAND"):
        return "command"

    return "conversation"


In [42]:
def conversation_node(state: AgentState):
    # use to convecte with user
    user_input = state.get("user_input", "")
    history = state.get("history", [])

    conversation_history = "\n".join(
        f"{'User' if isinstance(message, HumanMessage) else 'Agent TK'}: "
        f"{message.content}"
        for message in history
    )

    prompt = f"""
You are Agent.

Have a natural conversation with the user.

Conversation history:
{conversation_history}

Current user message:
{user_input}

Respond naturally and concisely.
Do not execute operating-system commands.
"""

    response = llm.invoke(prompt)

    answer = response.content.strip()

    history.append(
        AIMessage(content=answer)
    )

    speak(answer)

    return {
        "response": answer,
        "history": history
    }


In [43]:
def find_command_results(result):
    # function for rag_node
    command_results = []

    for item in result["results"]:
        text = item.get("raw_content", "")

        if any(keyword in text.lower() for keyword in [
            "command",
            "cmd",
            "command prompt",
            "powershell",
            "terminal",
            "run this",
            "type the following"
        ]):
            command_results.append({
                "title": item["title"],
                "url": item["url"],
                "content": text
            })

    return command_results

In [44]:
def rag_node(state: AgentState):
    # gets data from website
    user_input = state.get("user_input", "")

    tavily_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced")

    result = tavily_search.invoke({
        "query": f"Windows cmd command for: {user_input}",
        "include_domains": [
            "learn.microsoft.com"
        ]
    })

    urls = [
        item["url"]
        for item in result.get("results", [])
    ]

    tavily_extract = TavilyExtract()

    extracted = tavily_extract.invoke({
        "urls": urls[:3]
    })

    extracted = find_command_results(extracted)

    joined = chr(10).join(
        str(result)
        for result in extracted
    )

    return {
        "rag_results": joined
    }

In [45]:
def command_node(state: AgentState):
    # gets command

    user_input = state.get("user_input", "")
    rag_results = state.get("rag_results", "")

    prompt = f"""
You are a Windows CMD command generator.

Your ONLY task is to convert the USER REQUEST into the ONE
Windows CMD command required to perform that request.

USER REQUEST:
{user_input}

RETRIEVED WINDOWS DOCUMENTATION:
{rag_results}

IMPORTANT:
Use the retrieved documentation as supporting information.
Use your own knowledge of Windows CMD when necessary.

OUTPUT CONTRACT:
Your response MUST contain exactly ONE executable Windows CMD command
and NOTHING ELSE.

The first character of your response must be the first character
of the command.

The last character of your response must be the last character
of the command.

NEVER output:
- explanations
- reasoning
- analysis
- <think> or </think>
- introductions
- conclusions
- "Command:"
- "The command is:"
- "Here is the command:"
- Markdown
- backticks
- code fences
- comments
- bullet points
- numbered lists
- multiple commands
- alternative commands
- descriptions of the command

DO NOT explain what the command does.

DO NOT mention the user's request.

DO NOT mention the documentation.

DO NOT surround the command with any punctuation.

Examples of VALID output:
start microsoft.windows.camera:
notepad.exe
calc.exe
taskkill /F /IM notepad.exe

Examples of INVALID output:
The command is: notepad.exe
`notepad.exe`
Here is the command:
notepad.exe
The command opens Notepad using notepad.exe.

Return ONLY the executable Windows CMD command.
"""

    response = llm.invoke(prompt)

    command = response.content.strip()

    # remove thinking blocks
    command = re.sub(
        r"<think>.*?</think>",
        "",
        command,
        flags=re.IGNORECASE | re.DOTALL
    ).strip()

    # remove code fences
    command = re.sub(
        r"^```(?:cmd|bat|batch|windows)?\s*",
        "",
        command,
        flags=re.IGNORECASE
    )

    command = re.sub(
        r"\s*```$",
        "",
        command
    ).strip()

    # extract content from backticks
    matches = re.findall(
        r"`([^`]+)`",
        command,
        flags=re.DOTALL
    )

    if matches:
        command = max(matches, key=len).strip()

    # remove common command prefixes
    command = re.sub(
        r"^(?:command|cmd|windows command)\s*:\s*",
        "",
        command,
        flags=re.IGNORECASE
    ).strip()

    command = re.sub(
        r"^(?:the\s+)?command\s+is\s*:\s*",
        "",
        command,
        flags=re.IGNORECASE
    ).strip()

    # remove surrounding quotes
    command = command.strip().strip('"').strip("'").strip()

    # if explanation remains, try to extract a command-like line
    lines = [
        line.strip()
        for line in command.splitlines()
        if line.strip()
    ]

    if len(lines) > 1:

        command_patterns = [
            r"^(?:start|cmd|powershell|taskkill|tasklist|ipconfig|ping|tracert|nslookup|netstat|systeminfo|shutdown|"
            r"explorer|notepad|calc|mspaint|control|reg|sc|net|whoami|where|dir|cd|copy|move|del|erase|mkdir|rmdir|"
            r"echo|type|find|findstr|set|cls|title|color|hostname|ver|winver|msconfig|services\.msc|devmgmt\.msc|"
            r"diskmgmt\.msc|eventvwr\.msc|taskmgr\.exe|cmd\.exe|powershell\.exe|"
            r"[A-Za-z0-9_.-]+\.exe)"
        ]

        for line in lines:
            if any(
                re.search(
                    pattern,
                    line,
                    flags=re.IGNORECASE
                )
                for pattern in command_patterns
            ):
                command = line
                break

    # remove accidental surrounding punctuation
    command = command.strip().strip("`").strip()

    # final cleanup
    command = re.sub(
        r"^\s*(?:command|answer|output)\s*:\s*",
        "",
        command,
        flags=re.IGNORECASE
    ).strip()

    return {
        "command": command
    }

In [46]:
def verification_router(state: AgentState):

    command = state.get("command", "")

    prompt = f"""
You are the safety controller for Agent.

Determine whether the following Windows command requires
human confirmation before execution.

COMMAND:
{command}

Require confirmation for operations that can:
- delete files or folders
- terminate processes
- modify system settings
- install or uninstall software
- modify permissions
- change security configuration
- shutdown or restart the system
- overwrite important data
- perform potentially destructive operations

Do NOT require confirmation for harmless operations such as:
- opening an application
- opening a website
- opening Settings
- opening Calculator
- opening Notepad
- opening File Explorer
- reading information

Return ONLY:

YES

or

NO
"""

    response = llm.invoke(prompt)

    result = response.content.strip().upper()

    if "YES" in result:
        return "verification"

    return "execution"


In [47]:
def verification_node(state: AgentState):
    # verifys execution or cancel
    command = state.get("command", "")

    speak(
        f"I need your confirmation before executing this command: "
        f"{command}. Do you want me to proceed?"
    )

    response = listen()

    prompt = f"""
Determine whether the user is confirming or rejecting execution
of the command.

COMMAND:
{command}

USER RESPONSE:
{response}

Return ONLY one word:

YES

or

NO

Rules:
- YES only if the user clearly confirms execution.
- NO if the user rejects, cancels, refuses, or says not to proceed.
- If the response is ambiguous, return NO.
- Consider the meaning of the complete response, not individual keywords.

Return ONLY one word:

YES

or

NO

"""

    result = llm.invoke(prompt).content.strip().upper()

    if str.lower(result) == str.lower("YES"):
        decision = "execution"
    else:
        decision = "cancel"

    return {
        "verification_response": response,
        "verification_decision": decision
    }

In [48]:
def execution_node(state: AgentState):
    # executes 
    # gives litle responce
    command = state.get("command", "")

    print("\nExecuting:", command)

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )

        stdout = result.stdout.strip()
        stderr = result.stderr.strip()

        if result.returncode == 0:
            execution_result = (
                stdout
                if stdout
                else "Operation completed successfully."
            )
        else:
            execution_result = (
                f"Command failed.\n"
                f"{stderr or stdout}"
            )

    except subprocess.TimeoutExpired:
        execution_result = "The command timed out."

    except Exception as e:
        execution_result = f"Execution error: {str(e)}"

    prompt = f"""
The following Windows CMD command was executed:

{command}

Execution result:

{execution_result}

Give the user a short natural-language response describing
whether the requested operation was successful.

Do not generate another command.
"""

    response = llm.invoke(prompt)

    answer = response.content.strip()

    history = state.get("history", [])

    history.append(
        AIMessage(content=answer)
    )

    speak(answer)

    return {
        "execution_result": execution_result,
        "response": answer,
        "history": history
    }

In [49]:
def cancel_node(state: AgentState):
    # just appends

    message = "Operation cancelled."

    history = state.get("history", [])

    history.append(
        AIMessage(content=message)
    )

    speak(message)

    return {
        "response": message,
        "history": history
    }


In [50]:
def end_node(state: AgentState):
    # bye
    speak("Goodbye.")

    return state


In [51]:
graph = StateGraph(AgentState)

# nodes
graph.add_node("user", User_node)
graph.add_node("conversation", conversation_node)
graph.add_node("rag", rag_node)
graph.add_node("command", command_node)
graph.add_node("verification", verification_node)
graph.add_node("execution", execution_node)
graph.add_node("cancel", cancel_node)
graph.add_node("end", end_node)

# start
graph.add_edge(START, "user")

# routing
graph.add_conditional_edges(
    "user",
    route_request,
    {
        "end": "end",
        "command": "rag",
        "conversation": "conversation"
    }
)

# conversation
graph.add_edge("conversation", "user")

# rag
graph.add_edge("rag", "command")

# command
graph.add_conditional_edges(
    "command",
    verification_router,
    {
        "verification": "verification",
        "execution": "execution"
    }
)

# verification
graph.add_conditional_edges(
    "verification",
    lambda state: state["verification_decision"],
    {
        "execution": "execution",
        "cancel": "cancel"
    }
)

# execution
graph.add_edge("execution", "user")

# cancel
graph.add_edge("cancel", "user")

# end
graph.add_edge("end", END)

# compile
agent = graph.compile()

In [52]:
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	user(user)
	conversation(conversation)
	rag(rag)
	command(command)
	verification(verification)
	execution(execution)
	cancel(cancel)
	end(end)
	__end__([<p>__end__</p>]):::last
	__start__ --> user;
	cancel --> user;
	command -.-> execution;
	command -.-> verification;
	conversation --> user;
	execution --> user;
	rag --> command;
	user -.-> conversation;
	user -.-> end;
	user -. &nbsp;command&nbsp; .-> rag;
	verification -.-> cancel;
	verification -.-> execution;
	end --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [53]:
# history
initial_state: AgentState = {
    "history": []
}

# run
final_state = agent.invoke(initial_state)


Listening...
You: are you brother open folder


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6aa06059-4ac02e6349c4aedb6481bd64;71b98244-d35e-4398-8d64-26d0aaa47ecb)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [ ]:
final_state